In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
import sys

import arviz as az
from IPython.display import clear_output
import matplotlib
from matplotlib import pyplot as plt
import matplotlib_inline.backend_inline
import numpy as np
import pandas as pd
from twiga.core.plot.matplot_theme import configure_matlib_style

matplotlib_inline.backend_inline.set_matplotlib_formats("svg", "pdf", "retina")  # For export

sys.path.append("../src/")
import warnings

import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings("ignore")
from cycler import cycler
import scienceplots as splt

figure_path = "../figures/SmartGridPaper/"
az.style.use(["science", "arviz-doc", "tableau-colorblind10"])
nice_fonts = {
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
    "ytick.right": False,
    "axes.labelsize": 10,
    "font.size": 10,
    # Make the legend/label fonts a little smaller
    "legend.title_fontsize": 10,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    # Set line widths
    "axes.linewidth": 0.5,
    "grid.linewidth": 0.5,
    "lines.linewidth": 1.0,
    # Remove legend frame
    "legend.frameon": False,
    "axes.prop_cycle": cycler(
        color=[
            "#107591",
            "#00c0bf",
            "#f69a48",
            "#fdcd49",
            "#8da798",
            "#a19368",
            "#525252",
            "#a6761d",
            "#7035b7",
            "#cf166e",
        ]
    ),
}


configure_matlib_style(font_size=12, dpi=480, latex=True)
matplotlib.rcParams.update(nice_fonts)


HERE = Path.cwd()
if not (HERE / "fpqr_experiment.py").exists():
    HERE = HERE / "experiment" / "chapter-5-fpqr"
RUNS_DIR = HERE / "results" / "chapter_5_fpqr" / "runs"
OUT_DIR = HERE / "results" / "analysis"
FIG_DIR = OUT_DIR / "figures"

sys.path.insert(0, str(HERE))
from chapters.ch5_fpqr.analyse_results import discover_runs, load_csv
from chapters.ch5_fpqr.fpqr_experiment import SPEC  # noqa: E402

ALL_GROUPS = list(SPEC.reference_conditions)
TIMING_GROUPS = {"benchmark", "benchmark_nhits"}

In [ ]:
groups = ALL_GROUPS
datasets = ["MLVS-PT", "MLVS-UK"]

full, coverage = load_csv(groups, "ch5_fpqr_full.csv")

In [ ]:
groups

In [ ]:
full.columns

In [ ]:
df = full.copy()[(full["group"] == "benchmark")]
df.condition.unique()

In [ ]:
plot_df = df.assign(
    Head=df["condition"].replace(
        {
            "mlpqf": "FPQR",
            "qr": "QR",
            "laplace": "LAPLACE",
            "bnn": "BNN",
        }
    )
)

fig = sns.catplot(
    data=plot_df,
    y="cwe_stratified",
    x="dataset",
    hue="Head",
    sharex=False,
    margin_titles=True,
    height=2.5,
    aspect=1.25,
    legend=True,
    estimator="median",
    errorbar="sd",
    kind="bar",
    capsize=0.4,
    err_kws={"color": ".85", "linewidth": 0.5},
)
fig.set_axis_labels("Dataset", "CWE", fontsize=10)
plt.yticks(np.arange(0, 1.2, 0.2))
fig.set(ylim=(0, 1))